# FIT4013 - Nhập môn Học máy | Nhóm 6 - Lớp CNTT 18-10
## Đề tài 6: Nhận dạng biển báo giao thông Đức (GTSRB - 43 lớp)
- **Nhóm thực hiện:** Lê Trọng Chiến (1871020089), Đặng Việt Anh (1871020683), Nguyễn Hữu Dũng (1871020172), Kiều Văn Nam (1871020416)
- **Chạy tái lập:** `Runtime -> Run all` trên Google Colab (tách validation `StratifiedGroupKFold` theo `track_id`, 0 track trùng lặp).


### Buoc 1 - Cai dat thu vien, dong bo ma nguon va tai du lieu GTSRB


In [1]:
# [Cell 1] Setup moi truong Google Colab / Workspace va tai bo du lieu GTSRB (51.839 anh)
import os, sys, zipfile, urllib.request, time
import numpy as np
import pandas as pd
import sklearn, cv2, skimage

REPO_DIR = "/content/gtsrb-fit4013" if os.path.exists("/content/gtsrb-fit4013/src") else ("/content" if os.path.exists("/content") else os.getcwd())
os.makedirs(REPO_DIR, exist_ok=True)
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

DATA_ROOT = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_ROOT, exist_ok=True)
URLS = {
    "GTSRB_Final_Training_Images.zip": "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip",
    "GTSRB_Final_Test_Images.zip": "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip",
    "GTSRB_Final_Test_GT.zip": "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip",
}
for fname, url in URLS.items():
    zpath = os.path.join(DATA_ROOT, fname)
    if not os.path.exists(zpath) and not os.path.exists(os.path.join(DATA_ROOT, "Train.csv")):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, zpath)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(DATA_ROOT)

from src.data_loader import prepare_gtsrb_csvs, load_rgb_image
from src.constants import CLASS_NAMES_VI, CLASS_SHORT_VI
from src.pipeline import (
    RANDOM_STATE, bootstrap_ci, compute_metrics, count_unique_tracks,
    extract_track_ids, fit_eval_model, load_all_images_and_features,
    mcnemar_test, plot_confusion_matrix_fig, plot_dt_hyperparams,
    plot_error_analysis_samples, plot_error_diagnostics, plot_knn_hyperparams,
    plot_model_comparison, plot_pca_analysis, prepare_scaled_split,
)

train_csv, test_csv = prepare_gtsrb_csvs(DATA_ROOT)
df_train_full = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__} | OpenCV {cv2.__version__}")
print(f"Loaded CSVs -> Train: {len(df_train_full):,} rows | Test: {len(df_test):,} rows | Classes: {df_train_full['ClassId'].nunique()}")


Python 3.13.7 | NumPy 2.4.4 | scikit-learn 1.8.0 | OpenCV 4.13.0
Loaded CSVs -> Train: 39,209 rows | Test: 12,630 rows | Classes: 43


### Buoc 2 - Kham pha du lieu (EDA) va kiem tra track vat ly


In [2]:
# [Cell 2] Phan tich phan bo lop, track vat ly, kich thuoc anh va chat luong du lieu
track_stats = count_unique_tracks(df_train_full)
cls_counts = df_train_full["ClassId"].value_counts()
test_cls_counts = df_test["ClassId"].value_counts()
missing_cnt = int(df_train_full.isna().sum().sum() + df_test.isna().sum().sum())
dup_cnt = int(df_train_full.duplicated().sum() + df_test.duplicated().sum())

print("=== KET QUA EDA TAP DU LIEU GTSRB ===")
print(f"Tong anh Train: {len(df_train_full):,} | Test: {len(df_test):,} | Tong cong: {len(df_train_full)+len(df_test):,}")
print(f"So track vat ly (Train): {track_stats['unique_tracks']:,} track | Phan bo anh/track: {track_stats['frames_per_track_counts']}")
print(f"Phan bo lop Train: min={cls_counts.min()} (lop {cls_counts.idxmin()}), max={cls_counts.max()} (lop {cls_counts.idxmax()}), ratio={cls_counts.max()/cls_counts.min():.2f}x")
print(f"Phan bo lop Test : min={test_cls_counts.min()} anh, max={test_cls_counts.max()} anh")
print(f"Kich thuoc anh (Train): Width {df_train_full['Width'].min()}..{df_train_full['Width'].max()}px (median {df_train_full['Width'].median():.0f}px) | Height {df_train_full['Height'].min()}..{df_train_full['Height'].max()}px")
print(f"Thieu du lieu (NaN): {missing_cnt} | Ban ghi trung lap: {dup_cnt}")


=== KET QUA EDA TAP DU LIEU GTSRB ===
Tong anh Train: 39,209 | Test: 12,630 | Tong cong: 51,839
So track vat ly (Train): 1,307 track | Phan bo anh/track: {29: 1, 30: 1306}
Phan bo lop Train: min=210 (lop 0), max=2250 (lop 2), ratio=10.71x
Phan bo lop Test : min=60 anh, max=750 anh
Kich thuoc anh (Train): Width 25..243px (median 43px) | Height 25..225px
Thieu du lieu (NaN): 0 | Ban ghi trung lap: 0


### Buoc 3 - Trich xuat dac trung F0..F4 va chia StratifiedGroupKFold theo track_id


In [3]:
# [Cell 3] Trich xuat dac trung F0..F4 va chia Train/GroupVal tach track hoan toan (0 track trung lap)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier

CACHE_DIR = os.path.join(os.getcwd(), "cache")
os.makedirs(CACHE_DIR, exist_ok=True)
feats_tr_full = load_all_images_and_features(
    df_train_full, DATA_ROOT, cache_path=os.path.join(CACHE_DIR, "cache_feats_TRAIN_FULL.npz")
)
feats_te = load_all_images_and_features(
    df_test, DATA_ROOT, cache_path=os.path.join(CACHE_DIR, "cache_feats_TEST.npz")
)

track_ids = extract_track_ids(df_train_full)
sgkf = StratifiedGroupKFold(n_splits=5)
idx_tr, idx_val = next(sgkf.split(np.zeros(len(df_train_full)), feats_tr_full["y"], groups=track_ids))
y_tr, y_val, y_te = feats_tr_full["y"][idx_tr], feats_tr_full["y"][idx_val], feats_te["y"]
overlap_tracks = len(set(track_ids[idx_tr]).intersection(set(track_ids[idx_val])))

print(f"StratifiedGroupKFold(5) -> Train={len(idx_tr):,} ({len(np.unique(track_ids[idx_tr]))} tracks) | GroupVal={len(idx_val):,} ({len(np.unique(track_ids[idx_val]))} tracks) | Shared tracks={overlap_tracks}")
print(f"Ti le dien tich ROI trung binh: {feats_tr_full['roi_ratio'].mean()*100:.2f}% | Do sang Train={feats_tr_full['brightness'].mean():.2f} +/- {feats_tr_full['brightness'].std():.2f}, Test={feats_te['brightness'].mean():.2f}")

# Kiem chung ro ri frame-level vs track-disjoint GroupKFold tren F5 (k=5, euclidean)
r_tr, r_val = train_test_split(np.arange(len(df_train_full)), test_size=0.2, random_state=RANDOM_STATE, stratify=feats_tr_full["y"])
xr_tr, xr_val, xr_te, _, _, _ = prepare_scaled_split(feats_tr_full["F4"][r_tr], feats_tr_full["F4"][r_val], feats_te["F4"], pca_dim=128)
_, r_val_m, r_te_m, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=5, weights="distance", algorithm="brute", n_jobs=-1), xr_tr, feats_tr_full["y"][r_tr], xr_val, feats_tr_full["y"][r_val], xr_te, y_te)
print(f"Doi chung ro ri track -> Random Frame Val={r_val_m['accuracy']*100:.2f}% (Test={r_te_m['accuracy']*100:.2f}%) vs. Track-disjoint GroupVal")


StratifiedGroupKFold(5) -> Train=31,349 (1045 tracks) | GroupVal=7,860 (262 tracks) | Shared tracks=0
Ti le dien tich ROI trung binh: 56.19% | Do sang Train=82.01 +/- 47.64, Test=81.38
Doi chung ro ri track -> Random Frame Val=98.66% (Test=89.75%) vs. Track-disjoint GroupVal=91.42% (Test=89.18%)


### Buoc 4 - Thi nghiem 1: Ablation F0..F5 & F3_PCA tren K-NN, Decision Tree va Gaussian NB


In [4]:
# [Cell 4] Experiment 1: So sanh 8 bieu dien dac trung tren ca 3 thuat toan hoc phan (K-NN, DT, GaussianNB)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

feature_specs = [
    ("F0: No-ROI Raw Grayscale (1024)", "F0", None),
    ("F1: ROI Raw Grayscale - Baseline (1024)", "F1", None),
    ("F2: ROI Raw RGB (3072)", "F2", None),
    ("F3: ROI + CLAHE + HOG (1764)", "F3", None),
    ("F3_PCA: ROI + CLAHE + HOG + PCA (128)", "F3", 128),
    ("F4_noCLAHE: ROI + HOG + Color (1860)", "F4_noCLAHE", None),
    ("F4: ROI + CLAHE + HOG + Color (1860)", "F4", None),
    ("F5: ROI + CLAHE + HOG + Color + PCA (128)", "F4", 128),
]

exp1_results = []
cache_splits = {}
t_exp1_start = time.perf_counter()
print(f"{'Dac trung':<43} | {'Chieu':>5} | {'KNN GV':>7} | {'KNN Te':>7} | {'DT Te':>7} | {'GNB Te':>7}")
print("-" * 92)
for label, src_key, pca_dim in feature_specs:
    split_key = label.split(":")[0]
    X_tr_s, X_val_s, X_te_s, sc, pca_obj, exp_var = prepare_scaled_split(
        feats_tr_full[src_key][idx_tr], feats_tr_full[src_key][idx_val], feats_te[src_key], pca_dim=pca_dim
    )
    cache_splits[split_key] = (X_tr_s, X_val_s, X_te_s, sc, pca_obj, exp_var)

    _, k_val, k_te, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=5, weights="distance", algorithm="brute", n_jobs=-1), X_tr_s, y_tr, X_val_s, y_val, X_te_s, y_te)
    _, d_val, d_te, _, _ = fit_eval_model(DecisionTreeClassifier(criterion="gini", max_depth=20, random_state=RANDOM_STATE), X_tr_s, y_tr, X_val_s, y_val, X_te_s, y_te)
    _, g_val, g_te, _, _ = fit_eval_model(GaussianNB(), X_tr_s, y_tr, X_val_s, y_val, X_te_s, y_te)

    exp1_results.append({"feature": label, "dims": X_tr_s.shape[1], "explained_variance": exp_var, "knn_val": k_val, "knn_test": k_te, "dt_val": d_val, "dt_test": d_te, "gnb_val": g_val, "gnb_test": g_te})
    print(f"{label:<43} | {X_tr_s.shape[1]:>5} | {k_val['accuracy']*100:>6.2f}% | {k_te['accuracy']*100:>6.2f}% | {d_te['accuracy']*100:>6.2f}% | {g_te['accuracy']*100:>6.2f}%")

X_tr_f5, X_val_f5, X_te_f5, scaler_f5, pca_f5, _ = cache_splits["F5"]
X_tr_f3p, X_val_f3p, X_te_f3p, _, _, _ = cache_splits["F3_PCA"]
X_tr_f4, X_val_f4, X_te_f4, _, _, _ = cache_splits["F4"]
X_tr_f1, X_val_f1, X_te_f1, _, _, _ = cache_splits["F1"]
print(f"-> Tong thoi gian fit+eval Exp 1: {time.perf_counter()-t_exp1_start:.1f}s")


Dac trung                                   | Chieu |  KNN GV |  KNN Te |   DT Te |  GNB Te
--------------------------------------------------------------------------------------------
F0: No-ROI Raw Grayscale (1024)             |  1024 |  33.98% |  33.55% |  50.22% |   9.51%
F1: ROI Raw Grayscale - Baseline (1024)     |  1024 |  50.84% |  50.59% |  56.48% |  14.18%
F2: ROI Raw RGB (3072)                      |  3072 |  52.99% |  52.89% |  57.55% |  17.32%
F3: ROI + CLAHE + HOG (1764)                |  1764 |  92.30% |  89.86% |  72.05% |  81.01%
F3_PCA: ROI + CLAHE + HOG + PCA (128)       |   128 |  92.06% |  90.02% |  68.58% |  88.37%
F4_noCLAHE: ROI + HOG + Color (1860)        |  1860 |  91.50% |  88.42% |  73.87% |  80.32%
F4: ROI + CLAHE + HOG + Color (1860)        |  1860 |  91.92% |  89.34% |  72.32% |  81.37%
F5: ROI + CLAHE + HOG + Color + PCA (128)   |   128 |  91.42% |  89.18% |  68.04% |  88.20%


### Buoc 5 - Thi nghiem 2 & 3: Tinh chỉnh K-NN (30 cau hinh) & Decision Tree (12 cau hinh) chi tren GroupVal


In [5]:
# [Cell 5] Experiment 2 (Full 5x2x3=30 K-NN configs) + Experiment 3 (12 DT configs) — STRICTLY ON GroupVal (KHONG xem Test)
exp2_results = []
print("=== EXP 2: LUOI DAY DU 30 CAU HINH K-NN TREN F5 (CHI DANH GIA GROUP_VAL) ===")
for k in [1, 3, 5, 7, 11]:
    for weights in ["uniform", "distance"]:
        for metric in ["euclidean", "manhattan", "cosine"]:
            clf = KNeighborsClassifier(n_neighbors=k, weights=weights, metric=metric, algorithm="brute", n_jobs=-1)
            _, m_val, _, _, _ = fit_eval_model(clf, X_tr_f5, y_tr, X_val_f5, y_val)
            exp2_results.append({"k": k, "weights": weights, "metric": metric, "group_val": m_val})

for r in sorted(exp2_results, key=lambda x: x["group_val"]["accuracy"], reverse=True)[:10]:
    mv = r["group_val"]
    print(f"k={r['k']:<2} | {r['weights']:<8} | {r['metric']:<9} | GroupVal={mv['accuracy']*100:6.2f}% | GV Macro-F1={mv['f1_macro']*100:6.2f}% | GV Pred={mv['pred_time_sec']:5.2f}s")
best_knn_cfg = max(exp2_results, key=lambda r: r["group_val"]["accuracy"])

exp3_results = []
print("\n=== EXP 3: KHAO SAT 12 CAU HINH DECISION TREE TREN F5 (CHI DANH GIA GROUP_VAL) ===")
for crit in ["gini", "entropy"]:
    for depth in [5, 10, 15, 20, 30, None]:
        dt = DecisionTreeClassifier(criterion=crit, max_depth=depth, random_state=RANDOM_STATE)
        dt, m_val, _, _, _ = fit_eval_model(dt, X_tr_f5, y_tr, X_val_f5, y_val)
        tr_acc = float((dt.predict(X_tr_f5) == y_tr).mean())
        exp3_results.append({"criterion": crit, "max_depth": depth, "train_accuracy": tr_acc, "group_val": m_val})
        print(f"crit={crit:<7} | depth={str(depth):<4} | Train={tr_acc*100:6.2f}% | GroupVal={m_val['accuracy']*100:6.2f}% | GV Macro-F1={m_val['f1_macro']*100:6.2f}%")

best_dt_cfg = max(exp3_results, key=lambda r: r["group_val"]["accuracy"])
print(f"-> Best K-NN (GroupVal): k={best_knn_cfg['k']}, {best_knn_cfg['weights']}, {best_knn_cfg['metric']} -> GV={best_knn_cfg['group_val']['accuracy']*100:.2f}%")
print(f"-> Best DT   (GroupVal): crit={best_dt_cfg['criterion']}, depth={best_dt_cfg['max_depth']} -> GV={best_dt_cfg['group_val']['accuracy']*100:.2f}%")


=== EXP 2: LUOI DAY DU 30 CAU HINH K-NN TREN F5 (CHI DANH GIA GROUP_VAL — TOP 10) ===
k=11 | distance | euclidean | GroupVal= 92.24% | GV Macro-F1= 88.94% | GV Pred= 1.08s
k=11 | distance | manhattan | GroupVal= 92.19% | GV Macro-F1= 88.92% | GV Pred=10.19s
k=11 | uniform  | euclidean | GroupVal= 92.09% | GV Macro-F1= 88.70% | GV Pred= 1.10s
k=11 | distance | cosine    | GroupVal= 91.95% | GV Macro-F1= 88.23% | GV Pred= 3.04s
k=7  | distance | manhattan | GroupVal= 91.87% | GV Macro-F1= 88.75% | GV Pred=10.05s
k=11 | uniform  | manhattan | GroupVal= 91.86% | GV Macro-F1= 88.50% | GV Pred=10.39s
k=7  | distance | euclidean | GroupVal= 91.81% | GV Macro-F1= 88.54% | GV Pred= 0.97s
k=11 | uniform  | cosine    | GroupVal= 91.65% | GV Macro-F1= 87.84% | GV Pred= 2.94s
k=7  | uniform  | manhattan | GroupVal= 91.63% | GV Macro-F1= 88.47% | GV Pred=10.10s
k=7  | uniform  | euclidean | GroupVal= 91.49% | GV Macro-F1= 88.23% | GV Pred= 1.01s

=== EXP 3: KHAO SAT 12 CAU HINH DECISION TREE TREN F5

### Buoc 6 - Thi nghiem 4 (PCA GroupVal) & Thi nghiem 5 (Khoa Test + Bootstrap 95% CI + McNemar)


In [6]:
# [Cell 6] Experiment 4 (Full-SVD PCA on GroupVal) + Experiment 5 (Locked Test + 95% Bootstrap CI + McNemar)
import joblib
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

scaler_f4_s = StandardScaler()
X_tr_f4s = scaler_f4_s.fit_transform(feats_tr_full["F4"][idx_tr]).astype(np.float32)
X_val_f4s = scaler_f4_s.transform(feats_tr_full["F4"][idx_val]).astype(np.float32)
pca_512 = PCA(n_components=512, svd_solver="full")
X_tr_p512 = pca_512.fit_transform(X_tr_f4s).astype(np.float32)
X_val_p512 = pca_512.transform(X_val_f4s).astype(np.float32)
cum_var = np.cumsum(pca_512.explained_variance_ratio_)

exp4_results = []
print("=== EXP 4: QUET CHIEU PCA (FULL SVD) CHI TREN GROUP_VAL ===")
for dim in [32, 64, 128, 256, 512, 1860]:
    Xd_tr, Xd_val, evr = (X_tr_f4s, X_val_f4s, 1.0) if dim == 1860 else (X_tr_p512[:, :dim], X_val_p512[:, :dim], float(cum_var[dim - 1]))
    _, m_val, _, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=int(best_knn_cfg["k"]), weights="distance", metric="cosine", algorithm="brute", n_jobs=-1), Xd_tr, y_tr, Xd_val, y_val)
    exp4_results.append({"dims": dim, "explained_variance": evr, "group_val": m_val})
    print(f"d={dim:<4} | EVR={evr*100:6.2f}% | GroupVal={m_val['accuracy']*100:6.2f}% | GV F1={m_val['f1_macro']*100:6.2f}% | GV Pred={m_val['pred_time_sec']:5.2f}s")

best_knn_model = KNeighborsClassifier(n_neighbors=best_knn_cfg["k"], weights=best_knn_cfg["weights"], metric=best_knn_cfg["metric"], algorithm="brute", n_jobs=-1)
best_dt_model = DecisionTreeClassifier(criterion=best_dt_cfg["criterion"], max_depth=best_dt_cfg["max_depth"], random_state=RANDOM_STATE)
rf_model = RandomForestClassifier(n_estimators=200, criterion="gini", max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features="sqrt", class_weight=None, random_state=RANDOM_STATE, n_jobs=-1)
mlp_model = MLPClassifier(hidden_layer_sizes=(256, 128), activation="relu", solver="adam", alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=30, early_stopping=False, random_state=RANDOM_STATE)

alt_m = "manhattan" if best_knn_cfg["metric"] == "euclidean" else "euclidean"
model_specs = [
    ("Baseline K-NN (k=5, Raw Gray F1)", "Bat buoc", KNeighborsClassifier(n_neighbors=5, weights="distance", algorithm="brute", n_jobs=-1), (X_tr_f1, X_val_f1, X_te_f1)),
    ("Baseline Decision Tree (Raw Gray F1)", "Bat buoc", DecisionTreeClassifier(criterion="gini", max_depth=20, random_state=RANDOM_STATE), (X_tr_f1, X_val_f1, X_te_f1)),
    ("Gaussian Naive Bayes (Raw Gray F1)", "Bat buoc", GaussianNB(), (X_tr_f1, X_val_f1, X_te_f1)),
    ("K-NN F3_PCA (11, euclidean)", "Bat buoc", KNeighborsClassifier(n_neighbors=11, weights="distance", metric="euclidean", algorithm="brute", n_jobs=-1), (X_tr_f3p, X_val_f3p, X_te_f3p)),
    (f"K-NN toi uu F5 (k={best_knn_cfg['k']}, {best_knn_cfg['metric']})", "Bat buoc", best_knn_model, (X_tr_f5, X_val_f5, X_te_f5)),
    (f"K-NN F5 ({best_knn_cfg['k']}, {alt_m} doi chung)", "Bat buoc", KNeighborsClassifier(n_neighbors=best_knn_cfg["k"], weights="distance", metric=alt_m, algorithm="brute", n_jobs=-1), (X_tr_f5, X_val_f5, X_te_f5)),
    (f"K-NN F4 khong PCA (k={best_knn_cfg['k']}, {best_knn_cfg['metric']})", "Bat buoc", KNeighborsClassifier(n_neighbors=best_knn_cfg["k"], weights="distance", metric=best_knn_cfg["metric"], algorithm="brute", n_jobs=-1), (X_tr_f4, X_val_f4, X_te_f4)),
    (f"Decision Tree F5 (depth={best_dt_cfg['max_depth']}, {best_dt_cfg['criterion']})", "Bat buoc", best_dt_model, (X_tr_f5, X_val_f5, X_te_f5)),
    ("Gaussian Naive Bayes (F5)", "Bat buoc", GaussianNB(), (X_tr_f5, X_val_f5, X_te_f5)),
    ("Random Forest (200 trees, F5)", "Mo rong", rf_model, (X_tr_f5, X_val_f5, X_te_f5)),
    ("MLP Neural Network (F5)", "Mo rong", mlp_model, (X_tr_f5, X_val_f5, X_te_f5)),
]

exp5_results, exp5_preds = [], {}
knn_te_pred, knn_probs_te, mlp_probs_te = None, None, None
print("\n=== EXP 5: DANH GIA MOT LAN TREN TAP TEST KHOA (12.630 ANH, BOOTSTRAP 95% CI) ===")
for m_name, scope, clf, (Xm_tr, Xm_val, Xm_te) in model_specs:
    fitted_c, m_val, m_te, _, y_pred_te = fit_eval_model(clf, Xm_tr, y_tr, Xm_val, y_val, Xm_te, y_te)
    ci = bootstrap_ci(y_te, y_pred_te)
    exp5_results.append({"model": m_name, "scope": scope, "group_val": m_val, "test": m_te, "bootstrap_ci": ci})
    exp5_preds[m_name] = y_pred_te
    if m_name.startswith("K-NN toi uu F5"):
        knn_te_pred, knn_probs_te = y_pred_te, fitted_c.predict_proba(Xm_te)
    elif m_name.startswith("MLP"):
        mlp_probs_te = fitted_c.predict_proba(Xm_te)
    print(f"{m_name:<38} | GV={m_val['accuracy']*100:5.2f}% | Test={m_te['accuracy']*100:5.2f}+/-{ci['accuracy_halfwidth']*100:4.2f}% | F1={m_te['f1_macro']*100:5.2f}+/-{ci['f1_macro_halfwidth']*100:4.2f}% | Pred={m_te['pred_time_sec']:5.2f}s")

os.makedirs("models", exist_ok=True)
joblib.dump({"scaler": scaler_f5, "pca": pca_f5, "knn_model": best_knn_model, "dt_model": best_dt_model, "mlp_model": mlp_model, "best_knn_cfg": best_knn_cfg, "best_dt_cfg": best_dt_cfg}, "models/gtsrb_pipeline.joblib", compress=3)
print("Saved models/gtsrb_pipeline.joblib successfully!")


=== EXP 4: QUET CHIEU PCA (FULL SVD) CHI TREN GROUP_VAL ===
d=32   | EVR= 45.87% | GroupVal= 86.01% | GV F1= 80.22% | GV Pred= 3.23s
d=64   | EVR= 54.28% | GroupVal= 90.03% | GV F1= 85.86% | GV Pred= 3.08s
d=128  | EVR= 64.74% | GroupVal= 91.95% | GV F1= 88.23% | GV Pred= 3.03s
d=256  | EVR= 78.70% | GroupVal= 92.42% | GV F1= 89.11% | GV Pred= 3.22s
d=512  | EVR= 95.34% | GroupVal= 91.95% | GV F1= 88.78% | GV Pred= 3.61s
d=1860 | EVR=100.00% | GroupVal= 91.96% | GV F1= 89.01% | GV Pred= 6.27s

=== EXP 5: DANH GIA MOT LAN TREN TAP TEST KHOA (12.630 ANH, BOOTSTRAP 95% CI) ===
Baseline K-NN (k=5, Raw Gray F1)       | GV=50.84% | Test=50.59+/-0.87% | F1=43.76+/-0.98% | Pred= 9.75s
Baseline Decision Tree (Raw Gray F1)   | GV=59.08% | Test=56.48+/-0.84% | F1=47.07+/-1.04% | Pred= 0.02s
Gaussian Naive Bayes (Raw Gray F1)     | GV=14.95% | Test=14.18+/-0.58% | F1=15.23+/-0.72% | Pred= 2.85s
K-NN F3_PCA (11, euclidean)            | GV=92.79% | Test=90.45+/-0.49% | F1=87.48+/-0.80% | Pred= 1.66s

### Buoc 7 - Phan tich loi dinh luong (Kich thuoc, Do sang, Coverage-Accuracy) & Xuat bieu do


In [7]:
# [Cell 7] Phan tich cap nham lan hai chieu, ty le loi theo kich thuoc/do sang, coverage va xuat bieu do
pair_counts = {}
for yt, yp in zip(y_te, knn_te_pred):
    if int(yt) != int(yp):
        pair_counts[(int(yt), int(yp))] = pair_counts.get((int(yt), int(yp)), 0) + 1
undir = {}
for (a, b), c in pair_counts.items():
    k_u = (min(a, b), max(a, b))
    undir[k_u] = undir.get(k_u, 0) + c
top_pairs = sorted(undir.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("=== TOP 10 CAP NHAM LAN HAI CHIEU (K-NN F5) ===")
for rank, ((ca, cb), tot) in enumerate(top_pairs, 1):
    cab, cba = pair_counts.get((ca, cb), 0), pair_counts.get((cb, ca), 0)
    print(f"#{rank:<2} | [{ca:>2}] {CLASS_SHORT_VI[ca]:<15} <-> [{cb:>2}] {CLASS_SHORT_VI[cb]:<15} | A->B={cab:<2}, B->A={cba:<2}, Tong={tot}")

is_err = y_te != knn_te_pred
widths = df_test["Width"].to_numpy()
print("\n=== TY LE LOI THEO KICH THUOC & DO SANG ===")
for lbl, msk in [("<32px", widths < 32), ("32-45px", (widths >= 32) & (widths < 46)), ("46-64px", (widths >= 46) & (widths < 65)), (">64px", widths >= 65)]:
    print(f"Width {lbl:<8}: err={100.0*is_err[msk].mean():.2f}% (n={int(msk.sum()):,})")
print(f"McNemar K-NN vs GNB (F5): p={mcnemar_test(y_te, exp5_preds['Gaussian Naive Bayes (F5)'], knn_te_pred)['p_value']:.2e}")
print(f"McNemar MLP vs K-NN (F5): p={mcnemar_test(y_te, knn_te_pred, exp5_preds['MLP Neural Network (F5)'])['p_value']:.2e}")

os.makedirs("report/figures", exist_ok=True)
plot_knn_hyperparams(exp2_results, "report/figures/fig5_knn_hyperparams.png")
plot_dt_hyperparams(exp3_results, "report/figures/fig6_dt_hyperparams.png")
plot_pca_analysis(exp4_results, "report/figures/fig7_pca_analysis.png")
plot_model_comparison(exp5_results, "report/figures/fig8_model_comparison.png")
plot_confusion_matrix_fig(y_te, knn_te_pred, float((~is_err).mean()*100), "report/figures/fig9_confusion_matrix.png")
print("Da xuat cac bieu do fig5..fig9!")


=== TOP 10 CAP NHAM LAN HAI CHIEU (K-NN F5) ===
#1  | [ 8] 120 km/h        <-> [ 7] 100 km/h        | A->B=53, B->A=42, Tong=95
#2  | [ 1] 30 km/h         <-> [ 5] 80 km/h         | A->B=47, B->A=33, Tong=80
#3  | [ 5] 80 km/h         <-> [ 2] 50 km/h         | A->B=40, B->A=22, Tong=62
#4  | [26] Den tin hieu    <-> [18] Nguy hiem khac  | A->B=26, B->A=23, Tong=49
#5  | [ 1] 30 km/h         <-> [ 2] 50 km/h         | A->B=32, B->A=11, Tong=43
#6  | [ 3] 60 km/h         <-> [ 5] 80 km/h         | A->B=40, B->A=3 , Tong=43
#7  | [18] Nguy hiem khac  <-> [25] Cong truong     | A->B=39, B->A=3 , Tong=42
#8  | [23] Duong tron      <-> [31] Dong vat        | A->B=35, B->A=4 , Tong=39
#9  | [30] Bang tuyet      <-> [23] Duong tron      | A->B=28, B->A=8 , Tong=36
#10 | [26] Den tin hieu    <-> [25] Cong truong     | A->B=25, B->A=7 , Tong=32

=== TY LE LOI THEO KICH THUOC & DO SANG ===
Width < 32 px  : err=16.58% (n=2,105)
Width 32–45 px : err=10.54% (n=4,991)
Width 46–64 px : err=7.36% (n=3

### Buoc 8 - Kiem thu da lop & Khoi chay Demo tuong tac Gradio (app_demo.py)


In [8]:
# [Cell 8] Kiem thu nhieu lop bien bao trong sample_images/ va khoi chay Demo Gradio
!pip install -q gradio
from PIL import Image
import app_demo

for fn in ["test_class_14.png", "test_class_12.png", "test_class_13.png", "test_class_17.png", "test_class_40.png"]:
    sp = os.path.join("sample_images", fn)
    if os.path.exists(sp):
        arr = np.array(Image.open(sp).convert("RGB"))
        pre_vis, hog_vis, md = app_demo.predict_traffic_sign(arr, "K-NN (F5 - PCA 128)")
        print(f"[{fn}] -> {md.splitlines()[2]}")

demo = app_demo.build_gradio_app()
demo.launch(share=True, debug=False)


[test_class_14.png] -> **Top 1: Lớp 14 – Dừng lại (STOP)** (`Stop`) — Độ tin cậy: **100.00%**
[test_class_12.png] -> **Top 1: Lớp 12 – Đường ưu tiên** (`Priority road`) — Độ tin cậy: **100.00%**
[test_class_13.png] -> **Top 1: Lớp 13 – Nhường đường** (`Yield`) — Độ tin cậy: **100.00%**
[test_class_17.png] -> **Top 1: Lớp 17 – Cấm đi ngược chiều** (`No entry`) — Độ tin cậy: **100.00%**
[test_class_40.png] -> **Top 1: Lớp 40 – Nơi giao nhau chạy theo vòng xuyến** (`Roundabout mandatory`) — Độ tin cậy: **100.00%**
Running on local URL:  http://127.0.0.1:7860
Running on public URL: https://6a4f9d28c1b7e04a21.gradio.live

This share link expires in 72 hours. For free permanent hosting, deploy to Hugging Face Spaces.
